<h1> Bronze <h1>

<h2>Imports<h2>

<h3>Spark and Initialization<h3>

In [1]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Warsaw_Bus_Project") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .getOrCreate()

<h3>Imports<h3>

In [2]:
import os
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"
import pyspark.pandas as ps
import requests
from pyspark.sql.types import FloatType
api_key = os.getenv("WARSAW_API_KEY")

<h2>Bus_Stops Table<h2>

<h3>Calling Bus Stops Api<h3>

In [3]:
url1 = f"https://api.um.warszawa.pl/api/action/dbtimetable_get?id=ab75c33d-3a26-4342-b36a-6e5fef0a3ac3"
response = requests.get(url1,params={"apikey": api_key})
bus_stop_data = response.json()

<h3>Transfromations<h3>

In [ ]:
parsed_data = []
for row in bus_stop_data['result']:
    flat_row = {item['key'] : item['value'] for item in row['values']}
    parsed_data.append(flat_row)
bus_stops = spark.createDataFrame(parsed_data)
bus_stops = bus_stops\
    .withColumn('dlug_geo',
                bus_stops['dlug_geo']
                .cast(FloatType())) \
    .withColumn('szer_geo',
                bus_stops['szer_geo']
                .cast(FloatType()))
print(bus_stops)
print(bus_stops.head())

DataFrame[dlug_geo: float, id_ulicy: string, nazwa_zespolu: string, slupek: string, szer_geo: float, zespol: string]
Row(dlug_geo=20.963031768798828, id_ulicy='2349', nazwa_zespolu='PKP Rakowiec', slupek='05', szer_geo=52.1971549987793, zespol='4008')


<h3>Creating and writing to table<h3>